$$
\begin{array}{c}
{\Huge \text{CAUSAL INFERENCE LAB 1}}\\\\
{\Large \textbf{Abhishek Bakshi}} \\

{\large \textit{Center for Data Science, New York University}} \\\\

{\Large \text{Office Hours: Wednesdays, 12:30 - 1:30 pm}} \\\\

{\large \textit{September 11, 2026}}\\\\

\text{Materials prepared by: Abhishek Bakshi, Xiang Gao}

\end{array}
$$

---

<p align="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/1/1d/NYC_-_Washington_Square_Park_-_Arch.jpg/1280px-NYC_-_Washington_Square_Park_-_Arch.jpg">
</p>

<h1 style="font-size: 42px;">Introduction</h1>

---

<h1 style="font-size: 30px;">Today's Goals:</h1>

<ol style="font-size: 20px; line-height: 1.6;">
  <li>Python IDE setup</li>
  <li>Descriptive Questions vs Causal Questions</li>
  <li>Framing a Causal Inference Problem with the All-Causes Model</li>
  <li>Estimation in Real Data</li>
  <li>Identification and Estimation of Causal Parameters with Simulated Data</li>
</ol>

---

<style>
/* Large, consistent classroom typography for rendered Markdown. */
body,
.jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.markdown-body,
.vscode-body {
  font-family: Arial, Helvetica, sans-serif !important;
}

p, li, blockquote, td, th {
  font-size: 20px !important;
  line-height: 1.6 !important;
}

h1 {
  font-family: Arial, Helvetica, sans-serif !important;
  font-size: 42px !important;
  line-height: 1.25 !important;
  margin-top: 0.6em !important;
}

h2 {
  font-family: Arial, Helvetica, sans-serif !important;
  font-size: 30px !important;
  line-height: 1.3 !important;
  margin-top: 0.7em !important;
  border-bottom: 2px solid #4b4bb7;
  padding-bottom: 0.15em;
}

h3 {
  font-family: Arial, Helvetica, sans-serif !important;
  font-size: 25px !important;
  line-height: 1.35 !important;
  margin-top: 0.7em !important;
}

h4 {
  font-family: Arial, Helvetica, sans-serif !important;
  font-size: 22px !important;
  line-height: 1.4 !important;
}

table {
  font-size: 19px !important;
  line-height: 1.45 !important;
}

code, pre {
  font-size: 17px !important;
}

blockquote {
  border-left: 5px solid #4b4bb7 !important;
  padding: 0.3em 0.8em !important;
  color: inherit !important;
}

.MathJax, mjx-container {
  font-size: 115% !important;
}
</style>

## 1. VS Code Setup

---

### Python and Jupyter in VS Code

Useful installation resources:

- [Install VS Code](https://code.visualstudio.com/download)
- [Install Python](https://realpython.com/installing-python/)
- [Python extension for VS Code](https://marketplace.visualstudio.com/items?itemName=ms-python.python)
- [Jupyter extension for VS Code](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter)
- [Jupyter installation documentation](https://jupyter.org/install)

After opening this notebook in VS Code:

1. Look in the upper-right corner for **Select Kernel**.
2. Choose a Python installation.
3. Click the triangular run button beside the cell below, or press **Shift+Enter**.

In [ ]:
# A quick check that this notebook is connected to Python
import sys

print("Python is working!")
print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)


In [ ]:
# !pip install numpy pandas matplotlib

In [ ]:
# Import the libraries used in this lab
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 15,
    "axes.titlesize": 20,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

## 2. Descriptive Questions vs Causal Questions

---

### Quick activity: descriptive or causal?

1. Do students who attend more classes have higher grades?
2. What is the effect of regular attendance on a student's grade?
3. What proportion of summer-job applicants were arrested that summer?
4. Would providing a summer job reduce the probability of arrest?

<details>
<summary><strong>Answers</strong></summary>

1. Descriptive: it compares observed students.
2. Causal: it asks how grades change under an attendance intervention.
3. Descriptive: it summarizes observed arrests.
4. Causal: it contrasts potential outcomes with and without a summer job.

</details>

### What is the difference?



- Descriptive questions ask about the world **as it is observed**:
- Causal questions ask how the world **would change under a different state or intervention**:
<br></br>

| Descriptive Questions | Causal Questions |
|---|---|
| What is the unemployment rate in New York City? | What would the unemployment rate be if the minimum wage were higher? |
| How many people watched the World Cup final? | How many people would have watched the World Cup final if Lionel Messi had not played? |
| What is the average grade in this course? | What would students' grades be if they attended class more regularly? |

## 3. Framing a Causal Inference Problem with the All-Causes Model

---

| Component | What to specify |
|---|---|
| Causal question | What intervention and outcome are we studying? |
| Unit $i$ | Who or what could receive the intervention? |
| Target population | To whom should the answer apply? |
| State $S_i$ | What are the treatment and comparison states? |
| Outcome $Y_i$ | What is measured? |
| Potential outcomes | What are $Y_i(1)$ and $Y_i(0)$? |
| Causal estimand | What contrast answers the question? |
| Identification | Why might observed data reveal that contrast? |

### Class Example: summer youth employment

**Causal question:** What is the effect of receiving a government-provided summer job on being arrested that summer?

- **Unit $i$:** A particular youth who applied to the program.
- **Target population:** NYC youth who applied during the relevant program years.
- **$S_i=1$:** Youth $i$ receives a summer job.
- **$S_i=0$:** Youth $i$ does not receive a summer job.
- **$Y_i(s)=1$:** Youth $i$ would be arrested that summer under state $s$; otherwise $Y_i(s)=0$.
- **Potential outcomes:** $Y_i(1)$ and $Y_i(0)$.
- **Average treatment effect:**

$$
\operatorname{ATE}=\mathbb{E}[Y(1)-Y(0)].
$$

- **Identification idea:** Among applicants, job offers were allocated by lottery. Random assignment makes the lottery groups comparable on average.

### New Example: attendance and grades

Consider the question:

> What is the causal effect of regularly attending class on the final grade of students enrolled in this course?

1. What is the unit?
2. What is the target population?
3. What do $S_i=1$ and $S_i=0$ mean?
4. What do $Y_i(1)$ and $Y_i(0)$ mean?
5. What causal estimand would answer the question?
6. Why might comparing attending and non-attending students fail to identify it?

<details>
<summary><strong>Suggested framing</strong></summary>

- Unit: student $i$ enrolled in the course.
- Target population: students enrolled in this course.
- $S_i=1$: student $i$ attends regularly.
- $S_i=0$: student $i$ does not attend regularly.
- $Y_i(1)$: the final grade student $i$ would receive with regular attendance.
- $Y_i(0)$: the final grade student $i$ would receive without regular attendance.
- Estimand: $\mathbb{E}[Y(1)-Y(0)]$ in the target population.
- Identification concern: motivation, preparation, health, work obligations, and other factors may affect both attendance and grades.

</details>

### The All-Causes Model

The all causes model represents the outcome for unit $i$ as

$$
Y_i(S,U),
$$

where:

- $Y$ is the outcome of interest.
- $i$ identifies the unit of observation.
- $S$ is a possible state of the world or treatment.
- $U$ represents all other factors that influence the outcome for unit $i$.

For a binary treatment, $S\in\{0,1\}$, the two potential outcomes are

$$
Y_i(1,U) \quad \text{and} \quad Y_i(0,U).
$$

The individual causal effect of state 1 relative to state 0 is

$$
Y_i(1,U)-Y_i(0,U).
$$

### Applying the all causes model

For summer youth employment:

- $Y_i(s,u)$ is youth $i$'s arrest outcome under summer-job state $s$ and other factors $u$.
- $S_i=1$ means receiving the summer job; $S_i=0$ means not receiving it.
- $U_i$ could include prior criminal-justice contact, family circumstances, neighborhood conditions, age, or other causes of arrest.

The causal effect for youth $i$ is

$$
Y_i(1,U_i)-Y_i(0,U_i).
$$

The average treatment effect in the target population is

$$
\mathbb{E}[Y(1,U)-Y(0,U)].
$$

### The fundamental problem of causal inference

For each unit, we observe only the outcome associated with the state that actually occurred:

$$
Y_i =
\begin{cases}
Y_i(1,U_i), & S_i=1,\\
Y_i(0,U_i), & S_i=0.
\end{cases}
$$

We never observe both potential outcomes for the same unit at the same time. Therefore, we cannot directly calculate an individual causal effect from observed data.

This creates the central challenge:

> How can comparisons among different observed units reveal an average comparison between potential states?

### From a causal question to an answer

The course separates three tasks:

1. **Define the causal parameter:** What mathematical quantity answers the question?
2. **Identify the parameter:** Under what assumptions can it be written using observable quantities?
3. **Estimate the parameter:** How do we use a finite sample to calculate an estimate and express uncertainty?

The next section focuses on estimation of observable quantities. We will then return to identification using simulated randomized data.

## 4. Estimation in Real Data

---

### The 2022 General Social Survey

We use a small extract from the **2022 General Social Survey (GSS)**, conducted by NORC at the University of Chicago with principal funding from the National Science Foundation.

The original public-use data and documentation are available from the [GSS website](https://gss.norc.org/get-the-data.html). This file contains respondents aged 25-64 who reported that they were working full time, working part time, or temporarily unemployed.

Our causal question is:

Among working U.S. adults aged 25-64, what is the causal effect of having a bachelor's degree or higher on annual family income?

<details>
<summary><strong>Causal Framing</strong></summary>

- **Unit:** GSS respondent $i$.
- **Target population:** Working U.S. adults aged 25-64, subject to the limits of the survey and our sample restriction.
- **$S_i=1$:** Bachelor's degree or higher.
- **$S_i=0$:** Less than a bachelor's degree.
- **$Y_i$:** Inflation-adjusted annual family income.

</details>

In [ ]:
# Load the CSV stored in the same folder as this notebook
data_path = "gss_2022_education_income.csv"
gss = pd.read_csv(data_path)

print("Rows and columns:", gss.shape)
gss.head()


### What does a row represent? What does a column represent?

| Column | Meaning |
|---|---|
| `respondent_id` | Anonymous GSS respondent identifier |
| `age` | Age in years |
| `years_education` | Highest year of schooling completed |
| `highest_degree` | Highest degree category |
| `bachelors_or_more` | 1 for bachelor's/graduate degree, 0 otherwise |
| `annual_family_income` | GSS `CONINC`: inflation-adjusted family income derived from categorical income brackets using midpoints and imputations |
| `employment_status` | Reported labor-force status |
| `sex` | GSS sex category |
| `race` | GSS race category |

### EDA

In [ ]:
# Inspect variable types and missing values
print(gss.dtypes)

print("\nMissing values by column:")
gss.isna().sum()


In [ ]:
# Look at summary statistics for the numerical variables
gss[[
    "age",
    "years_education",
    "bachelors_or_more",
    "annual_family_income",
]].describe()


In [ ]:
# Distribution of observed family income
income = gss["annual_family_income"].dropna()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(income, bins=25, color="#4C78A8", edgecolor="white")
ax.axvline(income.mean(), color="#E45756", linewidth=2, label="Sample mean")
ax.set(
    title="Distribution of annual family income",
    xlabel="Annual family income (constant dollars)",
    ylabel="Number of respondents",
)
ax.legend()
plt.show()


### Expectations and their sample estimates

An expectation is a parameter of a probability distribution. We generally do not know the population expectation, so we estimate it using a sample.

| Quantity | Sample calculation |
|---|---|
| $\mathbb{E}[Y]$ | Mean of the observed `Y` values |
| $\mathbb{E}[Y\mid S=1]$ | Mean of `Y` among observations with `S == 1` |
| $\mathbb{E}[Y\mid S=0]$ | Mean of `Y` among observations with `S == 0` |
| $P(S=1)$ for binary $S$ | Mean of the 0/1 treatment indicator |

In this example:

- $S$ is `bachelors_or_more`.
- $Y$ is `annual_family_income`.

In [ ]:
# Estimate E[Y]: mean family income in the observed sample
mean_income = gss["annual_family_income"].mean()

print(f"Mean observed annual family income: ${mean_income:,.0f}")


In [ ]:
# For a binary variable, the mean estimates the proportion equal to 1
share_bachelors = gss["bachelors_or_more"].mean()

print(f"Share with a bachelor's degree or higher: {share_bachelors:.1%}")


### Conditional expectations

Before running the next cell, translate these expressions into words:

$$
\mathbb{E}[Y\mid S=1], \qquad \mathbb{E}[Y\mid S=0].
$$

What Pandas operation could calculate both quantities?

In [ ]:
# Estimate E[Y | S=0] and E[Y | S=1]
income_by_degree = gss.groupby("bachelors_or_more")["annual_family_income"].mean()
income_by_degree


In [ ]:
# Calculate the observed difference in conditional means
mean_no_degree = income_by_degree.loc[0]
mean_degree = income_by_degree.loc[1]
observed_difference = mean_degree - mean_no_degree

print(f"Mean without bachelor's or higher: ${mean_no_degree:,.0f}")
print(f"Mean with bachelor's or higher:    ${mean_degree:,.0f}")
print(f"Observed difference:               ${observed_difference:,.0f}")


In [ ]:
# Compare the observed income distributions by degree status
no_degree_income = gss.loc[gss["bachelors_or_more"] == 0, "annual_family_income"].dropna()

degree_income = gss.loc[gss["bachelors_or_more"] == 1, "annual_family_income"].dropna()

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.boxplot(
    [no_degree_income, degree_income],
    tick_labels=["No bachelor's", "Bachelor's or higher"],
    showfliers=False,
)
ax.set(
    title="Observed family income by degree status",
    ylabel="Annual family income (constant dollars)",
)
plt.show()


### Observed association is not automatically causation

We calculated the observable quantity

$$
\mathbb{E}[Y\mid S=1]-\mathbb{E}[Y\mid S=0].
$$

Our causal target is

$$
\mathbb{E}[Y(1)-Y(0)].
$$

These are different mathematical objects. The first compares two observed groups. The second compares two potential states for the target population.

#### Some questions

1. Does the observed difference prove that a bachelor's degree caused the income difference?
2. What other variables could affect both degree attainment and income?
3. Which relevant variables are absent from this dataset?
4. What research design or assumptions might make the comparison causal?

Possible elements of $U$ include family background, prior preparation, location, occupation, health, preferences, and labor-market opportunities.

## 5. Identification and Estimation with Simulated Data

---

### Potential outcomes in a simulated “perfect world”

With real data, we cannot observe both potential outcomes for the same person. In a simulation, we create the data-generating process, so we can temporarily inspect both states.

We return to the attendance-and-grades example:

- $S_i=1$: student $i$ attends regularly.
- $S_i=0$: student $i$ does not attend regularly.
- $Y_i(1)$ and $Y_i(0)$ are the student's two potential grades.
- Only $Y_i=Y_i(S_i)$ is observed.

Random assignment makes $S$ independent of the other causes represented by $U$. Under that identifying assumption, the observed difference in conditional means estimates the ATE.

In [ ]:
# Simulate students and both potential grades
rng = np.random.default_rng(42)

students = 1_000
attendance_effect = 10

baseline_grade = rng.normal(loc=75, scale=20, size=students)
Y0 = baseline_grade
Y1 = baseline_grade + attendance_effect

# Randomly assign regular attendance for this illustration
S = rng.binomial(n=1, p=0.5, size=students)

# The observed outcome selects exactly one potential outcome for each student
Y = np.where(S == 1, Y1, Y0)

simulated = pd.DataFrame({
    "student_id": np.arange(1, students + 1),
    "S": S,
    "Y0": Y0,
    "Y1": Y1,
    "Y_observed": Y,
})

simulated.sample(5)


In [ ]:
# In the simulated perfect world, we can calculate the true ATE
true_ate = (simulated["Y1"] - simulated["Y0"]).mean()

# In ordinary observed data, we estimate a difference between treatment groups
observed_means = simulated.groupby("S")["Y_observed"].mean()
estimated_difference = observed_means.loc[1] - observed_means.loc[0]

print(f"Observed means:\n{observed_means}")
print(f"True simulated ATE: {true_ate:.2f} grade points")
print(f"Observed difference in means: {estimated_difference:.2f} grade points")


### Why are the true and estimated values close but not identical?

Random assignment makes the two groups comparable **on average**, but a finite random sample can still contain chance imbalances. With larger samples, the observed difference generally becomes more stable around the true effect.

### Path forward

Future labs will repeatedly use the same workflow:

1. State the causal question.
2. Define units, population, states, outcomes, and estimand.
3. Inspect and visualize the available data.
4. State the assumptions connecting observed data to the estimand.
5. Estimate.
6. Interpret the result with its assumptions and limitations.